In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Model Training for Titanic Survival\n",
    "\n",
    "Training and evaluating multiple ML models"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.preprocessing import StandardScaler, LabelEncoder\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.ensemble import RandomForestClassifier, VotingClassifier\n",
    "from sklearn.metrics import accuracy_score, classification_report, confusion_matrix\n",
    "\n",
    "df = pd.read_csv('../data/titanic.csv')\n",
    "\n",
    "# Preprocessing\n",
    "df['Age'].fillna(df['Age'].median(), inplace=True)\n",
    "df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)\n",
    "\n",
    "# Feature engineering\n",
    "df['FamilySize'] = df['SibSp'] + df['Parch'] + 1\n",
    "df['IsAlone'] = (df['FamilySize'] == 1).astype(int)\n",
    "\n",
    "# Encode\n",
    "le = LabelEncoder()\n",
    "df['Sex_Encoded'] = le.fit_transform(df['Sex'])\n",
    "\n",
    "# Features\n",
    "features = ['Pclass', 'Sex_Encoded', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']\n",
    "X = df[features]\n",
    "y = df['Survived']\n",
    "\n",
    "# Scale\n",
    "scaler = StandardScaler()\n",
    "X[['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']] = scaler.fit_transform(\n",
    "    X[['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']]\n",
    ")\n",
    "\n",
    "# Split\n",
    "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n",
    "\n",
    "print(f\"Training: {X_train.shape[0]} samples\")\n",
    "print(f\"Testing: {X_test.shape[0]} samples\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Logistic Regression"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "lr = LogisticRegression(max_iter=1000, random_state=42)\n",
    "lr.fit(X_train, y_train)\n",
    "\n",
    "lr_pred = lr.predict(X_test)\n",
    "lr_acc = accuracy_score(y_test, lr_pred)\n",
    "\n",
    "print(f\"Logistic Regression Accuracy: {lr_acc:.4f}\")\n",
    "print(classification_report(y_test, lr_pred))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Random Forest"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "rf = RandomForestClassifier(n_estimators=100, random_state=42)\n",
    "rf.fit(X_train, y_train)\n",
    "\n",
    "rf_pred = rf.predict(X_test)\n",
    "rf_acc = accuracy_score(y_test, rf_pred)\n",
    "\n",
    "print(f\"Random Forest Accuracy: {rf_acc:.4f}\")\n",
    "print(classification_report(y_test, rf_pred))\n",
    "\n",
    "# Feature importance\n",
    "importance = pd.DataFrame({\n",
    "    'feature': features,\n",
    "    'importance': rf.feature_importances_\n",
    "}).sort_values('importance', ascending=False)\n",
    "\n",
    "print(\"\\nFeature Importance:\")\n",
    "print(importance)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Voting Ensemble"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "voting = VotingClassifier(\n",
    "    estimators=[('lr', lr), ('rf', rf)],\n",
    "    voting='soft'\n",
    ")\n",
    "voting.fit(X_train, y_train)\n",
    "\n",
    "voting_pred = voting.predict(X_test)\n",
    "voting_acc = accuracy_score(y_test, voting_pred)\n",
    "\n",
    "print(f\"Voting Ensemble Accuracy: {voting_acc:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Model Comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "results = pd.DataFrame({\n",
    "    'Model': ['Logistic Regression', 'Random Forest', 'Voting Ensemble'],\n",
    "    'Accuracy': [lr_acc, rf_acc, voting_acc]\n",
    "})\n",
    "\n",
    "print(\"\\n\" + \"=\"*40)\n",
    "print(\"MODEL COMPARISON\")\n",
    "print(\"=\"*40)\n",
    "print(results)\n",
    "\n",
    "best_model = results.loc[results['Accuracy'].idxmax(), 'Model']\n",
    "print(f\"\\n🏆 Best Model: {best_model} with {results['Accuracy'].max():.4f} accuracy\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}